# Canonical dense-backend example: coupled chromophore dimer

This notebook is a compact, reproducible guide to `LiouvilleSpectroscopySolver`. It constructs a coupled two-chromophore model, adds thermal Lindblad relaxation, computes third-order rephasing and non-rephasing spectra, resolves the six standard Liouville pathways, and plots the heterodyne-phased response.

The example intentionally uses only the dense backend. It also includes checks that catch two common mistakes: inactive collapse operators and inconsistent signs between total and pathway-resolved spectra.

## 1. Physical model

Each chromophore is represented as a two-level system. The four-state product basis contains the ground state, two singly excited states, and the doubly excited state. With lowering operators `a` and `b`, the Hamiltonian is

$$H = E_1 a^† a + E_2 b^† b + J(a^† b + b^† a).$$

The optical interaction is generated by the Hermitian transition dipole

$$μ = μ_1(a+a^†)+μ_2(b+b^†).$$

Relaxation and thermal excitation are described with four Lindblad channels. The solver expects pairs `(C, gamma)` and evaluates `gamma * D[C]`. Keeping the rate outside the jump matrix makes it explicit and prevents the operators from being silently interpreted as rate-free inputs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from LiouvilleSpectroscopyV6 import (
    LiouvilleSpectroscopySolver,
    SpectroscopyPlotter,
)

## 2. Units and conventions

This example uses natural units with `hbar = 1`:

- `E1`, `E2`, `J`, `Eta`, `kappa`, and the frequency grid are in eV.
- `tau2` is in inverse eV. To convert it to femtoseconds, use `tau2_fs = 0.6582119514 * tau2`.
- `kB = 8.617333262e-5 eV/K`.

The solver returns the complex third-order polarization response `P^(3)`. `SpectroscopyPlotter` applies a `+pi/2` detection phase by default, equivalent to plotting `1j * P^(3)`. Use `detection_phase=0` to inspect the raw polarization.

Spectrum arrays are indexed as `[omega1_index, omega3_index]`. Consequently, the plotting utilities place `omega3` on the horizontal axis and `omega1` on the vertical axis.

In [ ]:
def build_coupled_dimer_model(params):
    """Return H, the transition dipole, and thermal collapse channels.

    All operators are constructed in the site/product basis. Rates are
    supplied separately as (operator, gamma) pairs because the solver
    implements gamma * D[operator].
    """
    E1 = float(params["E1"])
    E2 = float(params["E2"])
    J = float(params["J"])
    mu1 = float(params.get("mu1", 1.0))
    mu2 = float(params.get("mu2", 1.0))
    kappa = float(params["kappa"])
    temperature = float(params["temperature"])
    kB = 8.617333262e-5  # eV/K

    identity = np.eye(2, dtype=complex)
    sigma_minus = np.array([[0, 1], [0, 0]], dtype=complex)

    a = np.kron(sigma_minus, identity)
    b = np.kron(identity, sigma_minus)
    a_dag = a.conj().T
    b_dag = b.conj().T

    n_a = a_dag @ a
    n_b = b_dag @ b
    hamiltonian = (
        E1 * n_a
        + E2 * n_b
        + J * (a_dag @ b + b_dag @ a)
    )
    dipole = mu1 * (a + a_dag) + mu2 * (b + b_dag)

    if temperature > 0:
        beta = 1.0 / (kB * temperature)
        n_thermal_1 = 1.0 / np.expm1(beta * E1)
        n_thermal_2 = 1.0 / np.expm1(beta * E2)
    else:
        n_thermal_1 = 0.0
        n_thermal_2 = 0.0

    collapse_channels = [
        (a, kappa * (n_thermal_1 + 1.0)),
        (b, kappa * (n_thermal_2 + 1.0)),
        (a_dag, kappa * n_thermal_1),
        (b_dag, kappa * n_thermal_2),
    ]
    return hamiltonian, dipole, collapse_channels

## 3. Model and solver parameters

`Eta` is an additional homogeneous broadening in the frequency-domain resolvent. It is distinct from the Lindblad rates. If both are nonzero, both contribute to the linewidth.

In [ ]:
model_params = {
    "E1": 2.00,
    "E2": 2.10,
    "J": 0.15,
    "mu1": 1.0,
    "mu2": 1.0,
    "kappa": 0.05,
    "temperature": 300.0,
}

solver_params = {
    "Eta": 0.01,
    "T": model_params["temperature"],
    "mu": 0.0,
    "backend": "dense",
    "parallel_backend": "threading",
    "n_jobs": 4,
    "blas_threads": 1,
    "spectrum_components": "both",
}

## 4. Choose the initial density matrix, then load the model

The initial state is optional. By default, `feed_model` constructs a normalized thermal density matrix from the Hamiltonian eigenenergies and the solver temperature `T` (or the ground state when `T=0`). Alternatively, define a density matrix outside the solver and pass it through `initial_density_matrix`. Its basis must be declared with `density_matrix_basis="site"` or `"eigen"`.

The example below defines the product-basis ground state externally but keeps the automatic thermal state as the default choice. After loading, `set_initial_density_matrix(rho, basis=...)` can replace the state, while `clear_initial_density_matrix()` restores the automatic thermal/default state.

`interaction_type="dipole"` tells the solver to split the Hermitian dipole into raising and lowering parts in the energy eigenbasis. Collapse operators are also transformed from the site basis during `feed_model`.

In [ ]:
# Optional externally defined state in the site/product basis.
rho_ground_site = np.zeros((4, 4), dtype=complex)
rho_ground_site[0, 0] = 1.0
USE_CUSTOM_INITIAL_STATE = False

H, mu_op, collapse_channels = build_coupled_dimer_model(model_params)

solver = LiouvilleSpectroscopySolver(solver_params)
solver.feed_model(
    H_model=H,
    interaction_op_array=mu_op,
    c_ops_raw=collapse_channels,
    interaction_type="dipole",
    initial_density_matrix=(
        rho_ground_site if USE_CUSTOM_INITIAL_STATE else None
    ),
    density_matrix_basis="site",
)

assert solver._active_backend == "dense"
assert len(solver.c_ops) == 4, "The Lindblad channels are not active."
print("Eigenenergies (eV):", solver.energies[0])
print("Initial density matrix in the eigenbasis:\n", solver.get_initial_density_matrix()[0])
print("Active Lindblad rates (eV):", [rate for _, rate in solver.c_ops])
print("Default pathways:")
for pathway in solver.pathway_summary():
    print(pathway)

## 5. Choose the pathway source: built-in definitions or UFSS

The solver provides the six canonical impulsive third-order pathways by default, so UFSS is not required for the standard R1-R6 calculation. This is the fastest and simplest choice.

Alternatively, UFSS can generate the Liouville diagrams from phase-discrimination conditions. `configure_standard_2d_pathways_with_ufss` asks UFSS for both `(-, +, +)` rephasing and `(+, -, +)` non-rephasing groups, translates the resulting `Ku/Kd/Bu/Bd` instructions, and installs them as the active pathways. UFSS must be installed, and the present frequency-domain solver accepts only canonical third-order pulse order `(0, 1, 2)`.

Set `USE_UFSS_PATHWAYS = True` to use UFSS. Call `solver.reset_pathways()` at any time to return to the internal defaults.

In [ ]:
USE_UFSS_PATHWAYS = False

if USE_UFSS_PATHWAYS:
    active_pathways = solver.configure_standard_2d_pathways_with_ufss(
        arrival_times=[0.0, 100.0, 200.0, 300.0],
    )
    pathway_source = "UFSS-generated"
else:
    solver.reset_pathways()
    active_pathways = solver.get_pathways()
    pathway_source = "built-in defaults"

print(f"Active pathway source: {pathway_source}")
for pathway in active_pathways:
    print(pathway.name, pathway.component, pathway.interactions)

## 6. Compute total and pathway-resolved spectra

The six default pathways are grouped as follows:

- Rephasing: `R1` (stimulated emission), `R2` (ground-state bleach), `R3` (excited-state absorption).
- Non-rephasing: `R4` (excited-state absorption), `R5` (stimulated emission), `R6` (ground-state bleach).

The ESA sign is already included in `R3` and `R4`. Therefore pathway spectra must be added directly; do not subtract ESA a second time.

In [ ]:
w_list = np.linspace(-2.7, 2.7, 121)
tau2 = 3.0  # eV^(-1), approximately 1.97 fs

spectra = solver.generate_2D_spectra(
    w_list,
    tau2=tau2,
    spectrum_components="both",
)
pathway_spectra = solver.generate_2D_pathways(
    w_list,
    tau2=tau2,
)

## 7. Internal consistency checks

The optimized dense scan and the sum of individually evaluated pathways should agree to numerical precision. These checks are useful whenever pathway definitions or prefactors are changed.

In [ ]:
rephasing_from_pathways = sum(pathway_spectra[name] for name in ("R1", "R2", "R3"))
nonrephasing_from_pathways = sum(pathway_spectra[name] for name in ("R4", "R5", "R6"))

rephasing_error = np.max(np.abs(spectra["rephasing"] - rephasing_from_pathways))
nonrephasing_error = np.max(np.abs(spectra["unrephasing"] - nonrephasing_from_pathways))

print(f"Maximum rephasing mismatch:     {rephasing_error:.3e}")
print(f"Maximum non-rephasing mismatch: {nonrephasing_error:.3e}")
assert np.allclose(spectra["rephasing"], rephasing_from_pathways)
assert np.allclose(spectra["unrephasing"], nonrephasing_from_pathways)

## 8. Plot the detected third-order signal

The default plotter phase is `+pi/2`, so the real row corresponds to the usual emitted-field/heterodyne-phased quadrature. Do not multiply the data by `1j` before passing them to the plotter, or the phase will be applied twice.

`zoom_bounds` is ordered as `(omega3_min, omega3_max, omega1_min, omega1_max)` because `omega3` is horizontal and `omega1` is vertical. Set `component` to `real`, `imag`, `abs`, or `all`.

In [ ]:
plotter = SpectroscopyPlotter(w_list)

plotter.plot_pathways_grid(
    pathways_dict=pathway_spectra,
    signal_type="rephasing",
    total_signal=spectra["rephasing"],
    component="real",
    zoom_bounds=(1.5, 2.7, -2.7, -1.5),
)

plotter.plot_pathways_grid(
    pathways_dict=pathway_spectra,
    signal_type="unrephasing",
    total_signal=spectra["unrephasing"],
    component="real",
    zoom_bounds=(1.5, 2.7, 1.5, 2.7),
)

To inspect the raw polarization instead of the detected quadrature, construct `SpectroscopyPlotter(w_list, detection_phase=0)`. If the experimental local-oscillator convention has the opposite sign, use `detection_phase=-np.pi/2`.

## 9. Verify the effect of the Lindblad rate

The following compact sweep rebuilds the model and solver for each `kappa`. The line cut uses `1j * P^(3)` explicitly because it is plotted directly with Matplotlib rather than through `SpectroscopyPlotter`. Increasing `kappa` should broaden and weaken spectral structure.

In [ ]:
kappa_values = [0.01, 0.05, 0.15]
w_sweep = np.linspace(-2.7, 2.7, 81)
fixed_w3_index = np.argmin(np.abs(w_sweep - 2.05))

fig, ax = plt.subplots(figsize=(7, 4.5))
for kappa in kappa_values:
    varied_params = {**model_params, "kappa": kappa}
    H_k, mu_k, channels_k = build_coupled_dimer_model(varied_params)
    solver_k = LiouvilleSpectroscopySolver({
        **solver_params,
        "T": varied_params["temperature"],
        "n_jobs": 1,
    })
    solver_k.feed_model(H_k, mu_k, channels_k, interaction_type="dipole")
    assert len(solver_k.c_ops) == 4

    response_k = solver_k.generate_2D_spectra(
        w_sweep,
        tau2=tau2,
        spectrum_components="rephasing",
        verbose=False,
    )["rephasing"]
    detected_k = 1j * response_k
    ax.plot(
        w_sweep,
        np.real(detected_k[:, fixed_w3_index]),
        label=fr"$\kappa={kappa:.2f}$ eV",
    )

ax.set_xlim(-2.7, -1.5)
ax.set_xlabel(r"$\omega_1$ (eV)")
ax.set_ylabel(r"Re$[iP^{(3)}]$")
ax.set_title(fr"Rephasing cut at $\omega_3={w_sweep[fixed_w3_index]:.2f}$ eV")
ax.legend()
fig.tight_layout()
plt.show()

## 10. Practical checklist

1. Use one consistent unit system for Hamiltonian energies, frequency axes, `Eta`, Lindblad rates, and waiting time.
2. Pass collapse channels as `(operator, gamma)` pairs and verify `len(solver.c_ops)`.
3. Rebuild or update the solver whenever model parameters or rates change.
4. Remember that the backend returns `P^(3)` while the plotter applies the detection phase.
5. Add signed pathways directly: `R1 + R2 + R3` and `R4 + R5 + R6`.
6. Compare optimized totals with pathway sums after changing solver internals.
7. Use `component="all"` during debugging, then select `real`, `imag`, or `abs` for presentation.